# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Leído de la imagen (fila, columna)
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10,   # entrega
            (2, 2): +2,    # estación de carga
            (3, 5): -10,   # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3,
            (4, 1): -3,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _move(self, state, action):
        """Aplica una acción determinista; si choca con borde/pared, se queda."""
        nxt = (state[0] + action[0], state[1] + action[1])
        return nxt if self.is_valid_state(nxt) else state

    def get_transition_probs(self, state, action):
        """Devuelve [(next_state, prob), ...] (las probabilidades suman 1)."""
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Desviaciones perpendiculares a la acción elegida
        if action in [(-1, 0), (1, 0)]:      # UP / DOWN -> desvía a LEFT / RIGHT
            left, right = (0, -1), (0, 1)
        else:                                 # LEFT / RIGHT -> desvía a UP / DOWN
            left, right = (-1, 0), (1, 0)

        p_main = 0.60 if state in self.slippery_states else 0.90
        p_side = (1.0 - p_main) / 2

        probs = {}
        for a, p in [(action, p_main), (left, p_side), (right, p_side)]:
            ns = self._move(state, a)
            probs[ns] = probs.get(ns, 0.0) + p
        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(p * V[ns] for ns, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for s in grid.states():
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    for k in range(1, max_iter + 1):
        delta = 0.0
        newV = dict(V)
        for s in grid.states():
            if grid.is_terminal(s):
                continue
            best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
            newV[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < threshold:
            return V, k
    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for s in grid.states():
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    for _ in range(max_iter):
        delta = 0.0
        newV = dict(V)
        for s in grid.states():
            if grid.is_terminal(s):
                continue
            newV[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria (siempre RIGHT)
    policy = {s: (0, 1) for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for it in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold)          # 2. evaluación
        new_policy = policy_improvement(grid, V)                # 3. mejora
        changes = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changes)
        policy = new_policy
        if changes == 0:                                        # 4. estabilidad
            break
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [19, 7, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Respuestas — Parte 5

**Lectura de la imagen (para validar el modelo):** START `(0,0)`; paredes `(0,3) (1,1) (2,4) (4,2)`; piso resbaloso `(1,2) (2,1) (3,3)`; entrega +10 `(0,5)`; carga +2 `(2,2)`; peligro mortal −10 `(3,5)`; peligros −3 `(1,4) (4,1)`. Total: 26 estados transitables.

**1. ¿Entrega +10 o carga +2?** Desde `START` el robot prefiere la **estación de carga +2**: la ruta óptima es `(0,0)→(0,1)→(0,2)→(1,2)→(2,2)`, solo 4 pasos. Llegar a la entrega implica más pasos y pasar por el peligro −3 en `(1,4)` o por la zona cercana al −10.

**2. ¿Por qué una recompensa menor puede ser óptima?** Porque lo que se maximiza es la **suma descontada de todo el camino**, no solo la recompensa final. Cada paso cuesta −1 y el descuento γ=0.9 reduce lo lejano. Una meta de +2 cercana puede valer más que un +10 lejano.

**3. ¿Dónde cambia la decisión el piso resbaloso?** En una celda resbalosa el robot solo hace lo que quiere el 60 % de las veces y el 40 % restante lo desvía de lado. Eso rebaja el valor de esos estados y hace que la política los atraviese solo cuando no hay mejor opción: `(1,2)` se usa porque es el paso directo hacia la carga. El Experimento B muestra que, aun empeorando mucho el piso, la decisión óptima no cambia aquí; solo baja el valor.

**4. Papel del costo por paso (−1):** es la "presión por llegar rápido". Penaliza caminar y hace que una meta cercana sea atractiva. Si se hace pequeño (Experimento A), merece la pena dar el rodeo hacia el +10.

**5. ¿Por qué T no es igual en todos los estados?** Porque la dinámica depende del **tipo de piso** del estado actual: 0.90/0.05/0.05 en piso normal y 0.60/0.20/0.20 en resbaloso. T debe recibir el estado como argumento y consultar `slippery_states`.

## Experimentos A, B, C y Bonus
Para cada uno se resuelve de nuevo con Value Iteration y se imprime la ruta más probable desde `START` y qué estados cambiaron de acción respecto a la política base.

In [7]:
def ruta_desde_start(g, pi, max_pasos=30):
    s = g.start; ruta = [s]
    for _ in range(max_pasos):
        if g.is_terminal(s): break
        s = g._move(s, pi[s]); ruta.append(s)
    return ruta, g.terminal_states.get(ruta[-1])

class WarehouseMDP_Resbaloso(WarehouseMDP):
    """Igual que WarehouseMDP, pero con prob. de movimiento deseado configurable en piso resbaloso."""
    def __init__(self, p_slip=0.60):
        super().__init__(); self.p_slip = p_slip
    def get_transition_probs(self, state, action):
        if self.is_terminal(state): return [(state, 1.0)]
        left, right = ((0,-1),(0,1)) if action in [(-1,0),(1,0)] else ((-1,0),(1,0))
        p_main = self.p_slip if state in self.slippery_states else 0.90
        p_side = (1 - p_main) / 2
        probs = {}
        for a, p in [(action, p_main), (left, p_side), (right, p_side)]:
            ns = self._move(state, a); probs[ns] = probs.get(ns, 0) + p
        return list(probs.items())

def resolver(nombre, g, pi_base=None):
    V, n = value_iteration(g); pi = extract_policy(g, V)
    ruta, term = ruta_desde_start(g, pi)
    print(f"--- {nombre} ---")
    print("V(START) =", round(V[g.start], 3), "| iteraciones:", n)
    print("Ruta:", ruta, "-> terminal:", f"{term:+d}")
    if pi_base:
        print("Estados con acción distinta a la base:", [s for s in pi if pi[s] != pi_base[s]])
    return pi

g0 = WarehouseMDP(); pi_base = resolver("BASE", g0)

gA = WarehouseMDP(); gA.living_reward = -0.1;  resolver("Experimento A (living_reward = -0.1)", gA, pi_base)
gB = WarehouseMDP_Resbaloso(0.40);             resolver("Experimento B (piso resbaloso 0.60 -> 0.40)", gB, pi_base)
gC = WarehouseMDP(); gC.gamma = 0.99;          resolver("Experimento C (gamma = 0.99)", gC, pi_base)

--- BASE ---
V(START) = -2.575 | iteraciones: 19
Ruta: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] -> terminal: +2
--- Experimento A (living_reward = -0.1) ---
V(START) = 1.649 | iteraciones: 25
Ruta: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] -> terminal: +10
Estados con acción distinta a la base: [(1, 0), (1, 2), (3, 2)]
--- Experimento B (piso resbaloso 0.60 -> 0.40) ---
V(START) = -2.706 | iteraciones: 20
Ruta: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)] -> terminal: +2
Estados con acción distinta a la base: []
--- Experimento C (gamma = 0.99) ---
V(START) = -1.456 | iteraciones: 23
Ruta: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)] -> terminal: +10
Estados con acción distinta a la base: [(1, 2)]


{(0, 0): (0, 1),
 (0, 1): (0, 1),
 (0, 2): (1, 0),
 (0, 4): (0, 1),
 (1, 0): (1, 0),
 (1, 2): (0, 1),
 (1, 3): (0, 1),
 (1, 4): (0, 1),
 (1, 5): (-1, 0),
 (2, 0): (0, 1),
 (2, 1): (0, 1),
 (2, 3): (-1, 0),
 (2, 5): (-1, 0),
 (3, 0): (0, 1),
 (3, 1): (0, 1),
 (3, 2): (-1, 0),
 (3, 3): (-1, 0),
 (3, 4): (0, -1),
 (4, 0): (-1, 0),
 (4, 1): (-1, 0),
 (4, 3): (-1, 0),
 (4, 4): (0, -1),
 (4, 5): (0, -1)}

In [8]:
# Bonus: barrido de living_reward para hallar el punto donde la política cambia de CARGA (+2) a ENTREGA (+10)
import numpy as np
anterior = None
for lr in np.arange(-1.0, 0.0001, 0.01):
    g = WarehouseMDP(); g.living_reward = round(float(lr), 2)
    V, _ = value_iteration(g); pi = extract_policy(g, V)
    _, term = ruta_desde_start(g, pi)
    if term != anterior:
        print(f"living_reward = {g.living_reward:+.2f}  ->  la política va a terminal {term:+d}")
        anterior = term

living_reward = -1.00  ->  la política va a terminal +2


living_reward = -0.79  ->  la política va a terminal +10


### Conclusiones de los experimentos

- **A (living_reward = −0.1):** la política cambia y desde `START` el robot **ya no va a la carga: va a la entrega +10** (ruta por `(1,2)→(1,3)→(1,4)→(1,5)→(0,5)`), aun pasando por el peligro −3. Caminar casi no cuesta, así que vale la pena el trayecto largo. Cambian las acciones en `(1,0)`, `(1,2)` y `(3,2)`.
- **B (piso resbaloso 0.60 → 0.40):** la **política no cambia**; solo empeora el valor (V(START) baja de −2.575 a ≈ −2.706). Los valores reflejan el riesgo adicional, pero la decisión óptima sigue siendo la misma.
- **C (γ = 0.99):** el robot valora más el futuro, y la política desde `START` pasa a buscar la **entrega +10**. Solo cambia la acción en `(1,2)` (de ↓ a →), justo la que decide entre carga y entrega. Es decir, con más paciencia el +10 distante pesa más. Nota: V(START) sigue siendo negativo porque el costo acumulado por paso pesa más al descontar menos.
- **Bonus:** el cambio ocurre aproximadamente en **living_reward ≈ −0.8**: por debajo (más negativo) conviene la carga +2, por encima conviene arriesgarse por la entrega +10.